# Chapter-11：把旅行多智能体发布成可调用的服务

> Last updated: 2026-11-23

## 学习目标

第 8 章构建了旅行多智能体系统（LangGraph 固定图 + A2A 子智能体 + 第三方 MCP 工具）。本章把这套系统封装成一个 MCP（Model Context Protocol）插件，让 Claude Code、Cursor、Codex 这类编码智能体也能直接调用它。重点理解：

1. MCP 是什么，为什么适合用来复用业务能力；
2. 把多智能体系统暴露成 MCP 工具的完整链路；
3. 如何把这个插件注册给三种编码智能体。

## 核心概念

MCP（Model Context Protocol）是一种开放协议，让“智能体/客户端”和“工具/服务器”之间用统一方式通信。一个 MCP 服务器把自己的能力声明成若干“工具”，客户端（比如 Claude Code）就能发现并调用这些工具。

与第 7 章的区别：第 7 章是让多智能体系统作为 MCP **客户端**，去调用天气、地图、航班等第三方 MCP 工具；本章方向相反，把多智能体系统自身发布成一个 MCP **服务器**，把整套旅行能力作为一个工具暴露给编码智能体。

本章做的，就是用 `mcp.server.fastmcp.FastMCP` 写一个 stdio MCP 服务器，在服务器内部包装第 8 章的 `LangGraphOrchestrator`，并暴露两个工具：

- `ask_travel_agent(query, thread_id?)`：返回旅行问题的最终答复文本；
- `ask_travel_agent_detailed(query, thread_id?)`：额外返回执行计划、子任务结果和 trace_id。

## 前置条件（隐含条件）

这些是必须满足、但容易被忽略的前提：

1. **5 个 A2A 子智能体服务必须已经在运行**（端口 9031~9035）。中心编排器通过 A2A 调用子智能体，所以 MCP 被调用前，这些服务需要先启动。
2. **第 8 章已安装**：`cd Chapter-8 && pip install -e .`。MCP 服务器会 `import travel_multi_agent`。
3. **已配置大模型 API key**：仓库根 `.env` 里有 `DASHSCOPE_API_KEY`（或 `OPENAI_API_KEY`）。
4. **第三方 MCP 工具需要 Node.js**：天气（weatherapi-mcp）、地图/酒店/餐厅（百度地图 MCP）、航班（飞常准 Aviation MCP）都通过 `npx` 启动，因此本机需要安装 Node.js，并在 `.env` 配置 `WEATHERAPI_KEY`、`BAIDU_MAP_API_KEY`、`VARIFLIGHT_API_KEY`。
5. **stdio 传输不能污染 stdout**：MCP 用 stdout 传协议消息，所以服务器里把 OTEL 输出关掉、日志压到 WARNING（`travel_agent_server.py` 已处理）。

In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_BOOK_ROOT = _HERE if (_HERE / 'Chapter-8').is_dir() else _HERE.parent
_CHAPTER8 = _BOOK_ROOT / 'Chapter-8'
if str(_CHAPTER8) not in sys.path:
    sys.path.insert(0, str(_CHAPTER8))

mcp_file = _BOOK_ROOT / 'Chapter-11' / 'travel_agent_server.py'
print('MCP 服务器文件存在:', mcp_file.is_file())

try:
    import travel_multi_agent  # noqa: F401
    print('travel_multi_agent 可导入: True')
except Exception as exc:
    print('travel_multi_agent 可导入: False ->', exc)

from llm import model_configured
print('模型 API key 已配置:', model_configured())

## 步骤一：启动 5 个 A2A 子智能体服务

每个领域智能体是一个独立进程。推荐的方式是运行 `Chapter-8/travel_multi_agent.ipynb` 里的“启动 5 个 A2A 子智能体服务”Cell；或在独立终端逐个执行：

```bash
cd Chapter-8
python -m travel_multi_agent.a2a.server --agent-card travel_multi_agent/agents/agent_cards/weather_agent.json --host 127.0.0.1 --port 9031 --public-url http://127.0.0.1:9031/
```

其余服务以此类推：`hotel_agent.json`(9032)、`restaurant_agent.json`(9033)、`flight_agent.json`(9034)、`itinerary_agent.json`(9035)。

## 步骤二：MCP 服务器如何包装多智能体

核心代码在 `Chapter-11/travel_agent_server.py`，要点如下：

```python
from mcp.server.fastmcp import FastMCP
from travel_multi_agent.orchestration.fixed_graph.orchestrator import LangGraphOrchestrator

mcp = FastMCP("travel-multi-agent")
_orchestrator = None

def _get_orchestrator():
    global _orchestrator
    if _orchestrator is None:
        _orchestrator = LangGraphOrchestrator(enable_memory=True)  # 惰性创建，进程内复用
    return _orchestrator

@mcp.tool()
async def ask_travel_agent(query: str, thread_id: str | None = None) -> str:
    result = await _get_orchestrator().process_request(query, thread_id=thread_id)
    return result.get("final_response") or ""
```

要点：MCP 进程是常驻的，所以编排器只创建一次、进程内复用；工具函数里直接调用 `process_request`，把整个多智能体系统封装为一个工具。

## 步骤三：注册给三种编码智能体

三种客户端都用“command + args”的 stdio 方式启动这个服务器，只是配置文件不同。下面把 `command` 替换为你的 python 绝对路径。

**Claude Code**：在项目根放 `.mcp.json`（Claude Code 启动时自动读取），或执行 `claude mcp add travel-agent -- python路径 script路径`。

```json
{
  "mcpServers": {
    "travel-agent": {
      "command": "E:\\Book\\agent-systems-book\\.venv\\Scripts\\python.exe",
      "args": ["Chapter-11/travel_agent_server.py"]
    }
  }
}
```

**Cursor**：在项目根放 `.cursor/mcp.json`，内容同上（同样是 `mcpServers` 对象）。

**Codex**：在全局配置 `~/.codex/config.toml` 里加：

```toml
[mcp_servers.travel-agent]
command = "E:\\Book\\agent-systems-book\\.venv\\Scripts\\python.exe"
args = ["Chapter-11/travel_agent_server.py"]
```

注意：`.mcp.json` / `.cursor/mcp.json` 里的相对路径相对项目根解析；Codex 的 `config.toml` 是全局文件，建议 `args` 用绝对路径。

## 步骤四：调用 MCP 工具

注册并重启客户端后，在 Claude Code / Cursor / Codex 的对话里即可由编码智能体调用工具，例如：

```
帮我查一下上海明天天气（用 travel-agent）
```

编码智能体会调用 `ask_travel_agent(query="查询上海明天天气")`，服务器内部执行第 8 章的完整编排（预调查 → 拆解 → 依赖分析 → 路由 → 并行调用子智能体 → 聚合），最终返回一段旅行答复；`ask_travel_agent_detailed` 则额外返回执行计划、子任务结果和 trace_id，便于调试。

## 小结

本章把第 8 章的旅行多智能体系统封装为一个 stdio MCP 服务器，对外暴露 `ask_travel_agent` / `ask_travel_agent_detailed` 两个工具，并注册给 Claude Code / Cursor / Codex 使用。第 7 章是让多智能体系统作为 MCP 客户端去调用第三方工具，本章方向相反：把多智能体系统本身发布为可供编码智能体调用的 MCP 服务器。使用时的关键约束是：调用前先启动 5 个 A2A 子智能体服务，并保证 API key 与 Node.js 依赖就绪。